# Sentiment Analysis (Naive Bayes)

**InsightForge Business Analytics Studio — Python Analyst Toolkit**

## Purpose
Train a simple, explainable text classifier on customer review text and evaluate how well it predicts sentiment (positive / neutral / negative).

## Methodology
1. Load `reviews.csv` and drop rows with missing `reviewText`.
2. Clean text: lowercase, strip punctuation/extra whitespace.
3. Split into train/test sets (`sklearn.model_selection.train_test_split`, stratified by label, 75/25 split, fixed random_state for reproducibility).
4. Vectorize with `TfidfVectorizer` (unigrams + bigrams, English stopwords removed).
5. Train a `MultinomialNB` classifier -- a standard, fast, interpretable baseline for text classification.
6. Evaluate on the held-out test set: accuracy, precision/recall/F1 (`classification_report`), and a plotted confusion matrix.
7. Inspect the most-informative terms per class from the fitted model's log probabilities.

## IMPORTANT: how the training labels were produced (read this first)
`data_generator.py` builds each synthetic review by first picking a sentiment bucket (`positive`/`neutral`/`negative`) and then sampling a template sentence *from that bucket*. The bucket that was used to generate the sentence is saved directly as `sentimentLabel` in `reviews.csv`, and that is exactly the label this notebook trains on.

**This is a demo-toolkit shortcut, not production-grade labeled data.** In a real project, sentiment labels come from actual human annotation (or a validated existing labeling process), not from the same generative process that created the text. Because the labels here are perfectly consistent with the templates that produced them, this notebook's accuracy numbers will likely look better than what you'd see on real, messy, human-written reviews -- the model is partly learning template phrasing, not just sentiment.

## Assumptions
- English-language review text only.
- Three-class sentiment (positive/neutral/negative); no aspect-based or fine-grained (1-5 star) sentiment modeling.

## Model Limitations
- **Small, synthetic training set.** A few thousand template-derived reviews is far smaller and far less varied than production sentiment datasets.
- **Template artifacts.** Because review text is drawn from a fixed set of ~20 sentence templates, the model can learn to recognize template phrasing rather than generalizable sentiment language. It will very likely perform worse on real, organically-written reviews than the metrics below suggest.
- **Not clinically/production validated.** This notebook demonstrates a full pipeline for a business-analytics toolkit; it is not a validated NLP product and should not be used to make consequential decisions without retraining on real, properly labeled data.
- **English-only.** No multilingual handling.
- **Label circularity**, as described above -- treat all metrics in this notebook as an illustration of the pipeline mechanics, not a real-world accuracy estimate.

In [1]:
import os
import re
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

TOOLKIT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, TOOLKIT_DIR)
from clean_data import load_csv

OUTPUT_DIR = os.path.join(TOOLKIT_DIR, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)
RANDOM_STATE = 42

In [2]:
reviews_path = os.path.join(TOOLKIT_DIR, "sample_data", "reviews.csv")
reviews = load_csv(reviews_path)

required_cols = {"reviewText", "sentimentLabel"}
if not required_cols.issubset(reviews.columns):
    print(f"Sentiment analysis unavailable: missing column(s) {sorted(required_cols - set(reviews.columns))}")
    reviews_clean = pd.DataFrame()
else:
    before = len(reviews)
    reviews_clean = reviews.dropna(subset=["reviewText", "sentimentLabel"]).copy()
    print(f"Dropped {before - len(reviews_clean)} rows with missing review text/label; {len(reviews_clean)} remain.")
    display(reviews_clean[["reviewText", "sentimentLabel"]].head())

Dropped 6 rows with missing review text/label; 2045 remain.


,reviewText,sentimentLabel
0,"Average Wireless Charger Pad, nothing special ...",neutral
1,"The Automatic Feeder works as described, no ma...",neutral
2,"Absolutely love this Pet Travel Carrier, it ex...",positive
3,"Not bad, not great. Meets basic expectations. ...",neutral
4,"Excellent value for the price, very happy with...",positive


## Text Cleaning

In [3]:
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

if not reviews_clean.empty:
    reviews_clean["cleanText"] = reviews_clean["reviewText"].apply(clean_text)
    display(reviews_clean[["reviewText", "cleanText", "sentimentLabel"]].head())

,reviewText,cleanText,sentimentLabel
0,"Average Wireless Charger Pad, nothing special ...",average wireless charger pad nothing special b...,neutral
1,"The Automatic Feeder works as described, no ma...",the automatic feeder works as described no maj...,neutral
2,"Absolutely love this Pet Travel Carrier, it ex...",absolutely love this pet travel carrier it exc...,positive
3,"Not bad, not great. Meets basic expectations. ...",not bad not great meets basic expectations pac...,neutral
4,"Excellent value for the price, very happy with...",excellent value for the price very happy with ...,positive


## Train/Test Split and Vectorization

In [4]:
if not reviews_clean.empty:
    X_train_text, X_test_text, y_train, y_test = train_test_split(
        reviews_clean["cleanText"],
        reviews_clean["sentimentLabel"],
        test_size=0.25,
        random_state=RANDOM_STATE,
        stratify=reviews_clean["sentimentLabel"],
    )
    print(f"Train size: {len(X_train_text)}, Test size: {len(X_test_text)}")

    vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english", min_df=2)
    X_train = vectorizer.fit_transform(X_train_text)
    X_test = vectorizer.transform(X_test_text)
    print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")

Train size: 1533, Test size: 512


Vocabulary size: 585


## Train the Naive Bayes Classifier

In [5]:
if not reviews_clean.empty:
    model = MultinomialNB()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print("Model trained.")

Model trained.


## Evaluation

In [6]:
if not reviews_clean.empty:
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy:.3f}")
    print()
    report = classification_report(y_test, y_pred)
    print(report)
    with open(os.path.join(OUTPUT_DIR, "03_classification_report.txt"), "w") as f:
        f.write(f"Accuracy: {accuracy:.3f}\n\n{report}")
    print("Saved output/03_classification_report.txt")

Accuracy: 0.996

              precision    recall  f1-score   support

    negative       1.00      0.98      0.99       103
     neutral       1.00      1.00      1.00       127
    positive       0.99      1.00      1.00       282

    accuracy                           1.00       512
   macro avg       1.00      0.99      1.00       512
weighted avg       1.00      1.00      1.00       512

Saved output/03_classification_report.txt


In [7]:
if not reviews_clean.empty:
    labels = sorted(reviews_clean["sentimentLabel"].unique())
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(6, 5))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title("Confusion Matrix: Sentiment Classifier")
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "03_confusion_matrix.png"), dpi=150)
    plt.close(fig)
    print("Saved output/03_confusion_matrix.png")

Saved output/03_confusion_matrix.png


## Most Informative Terms per Class

In [8]:
if not reviews_clean.empty:
    feature_names = np.array(vectorizer.get_feature_names_out())
    top_n = 12
    informative = {}
    for class_idx, class_label in enumerate(model.classes_):
        log_probs = model.feature_log_prob_[class_idx]
        top_idx = np.argsort(log_probs)[-top_n:][::-1]
        informative[class_label] = feature_names[top_idx].tolist()

    informative_df = pd.DataFrame(informative)
    informative_df.to_csv(os.path.join(OUTPUT_DIR, "03_informative_terms.csv"), index=False)
    print("Saved output/03_informative_terms.csv")
    display(informative_df)

Saved output/03_informative_terms.csv


,negative,neutral,positive
0,weeks,consider,purchase
1,weeks frustrating,time,price
2,stopped working,fine,quality
3,stopped,consider alternatives,fast
4,working,alternatives,stars complaints
5,frustrating,alternatives time,stars
6,working weeks,decent price,complaints
7,money wouldn,decent,shipping
8,wouldn recommend,price consider,value
9,wouldn,does,excellent value


## Summary
This notebook wrote `03_classification_report.txt`, `03_confusion_matrix.png`, and `03_informative_terms.csv` to `output/`.

**Re-read the Model Limitations section above before quoting these metrics anywhere** -- they describe pipeline mechanics on synthetic, template-derived data, not real-world sentiment-classification accuracy.